In [ ]:
import nltk

import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


In [ ]:
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download("stopwords")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

verb_codes = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}

In [ ]:
cs_conferences = pd.read_pickle("/workspaces/VRA/conference_rec/wikicfp_cs.pkl")


In [ ]:
cs_rec = cs_conferences[["Conference Title", "WikiCFP Tags", "Conference Description"]]
cs_rec.columns = ["title", "tags", "description"]
cs_rec = cs_rec.set_index("title")
cs_rec["soup"] = cs_rec["tags"] + " " + cs_rec["description"]

In [ ]:
def preprocess_sentences(text):
    text = text.lower()
    temp_sent = []
    words = nltk.word_tokenize(text)
    tags = nltk.pos_tag(words)
    for i, word in enumerate(words):
        if tags[i][1] in verb_codes:
            lemmatized = lemmatizer.lemmatize(word, 'v')
        else:
            lemmatized = lemmatizer.lemmatize(word)
        if lemmatized not in stop_words and lemmatized.isalpha():
            temp_sent.append(lemmatized)
            
    finalsent = ' '.join(temp_sent)
    finalsent = finalsent.replace("n't", " not")
    finalsent = finalsent.replace("'m", " am")
    finalsent = finalsent.replace("'s", " is")
    finalsent = finalsent.replace("'re", " are")
    finalsent = finalsent.replace("'ll", " will")
    finalsent = finalsent.replace("'ve", " have")
    finalsent = finalsent.replace("'d", " would")
    return finalsent

cs_rec["processed_soup"] = cs_rec["soup"].apply(preprocess_sentences)


In [ ]:
tfidfvec = TfidfVectorizer()
tfidf_model = tfidfvec.fit_transform(cs_rec["processed_soup"])
cos_sim = linear_kernel(tfidf_model, tfidf_model)

In [ ]:
indices = pd.Series(cs_rec.index)

def recommendations(title, cosine_sim = cos_sim):
    recommended_conferences = []
    index = indices[indices == title].index[0]
    similarity_scores = pd.Series(cosine_sim[index]).sort_values(ascending = False)
    top_10_conferences = list(similarity_scores.iloc[1:11].index)
    for i in top_10_conferences:
        recommended_conferences.append(list(cs_rec.index)[i])
    return recommended_conferences

In [ ]:
recommendations("CLOUD 2021 : 10th International Conference on Cloud Computing: Services and Architecture")

In [ ]:
recommendations("ECIJ 2021 : Electrical & Computer Engineering: An International Journal")